## Inferring the laterality from CFI_Keypoints model output

In [ ]:
from eyened_orm import Database, Modality, ImageInstance, Project

database = Database()

projectname = "" # insert your project name here
## Set to True to update the database value
update_database = False

In [2]:
from tqdm import tqdm

with database.get_session() as session:
    all_images = ImageInstance.where(
        session,
        (Project.ProjectName == projectname)
        & (ImageInstance.Modality == Modality.ColorFundus),
    )
    print(f"Found {len(all_images)} color fundus images")

    for image in tqdm(all_images):
        cfi_keypoints = image.get_attribute_value(producing_model_name="CFI_Keypoints")
        if cfi_keypoints is None:
            print("No CFI_Keypoints found for image ", image.ImageInstanceID)
            continue
        laterality = image.infer_laterality_from_keypoints(cfi_keypoints)
        etdrs_field = image.infer_ETDRS_field_from_keypoints(cfi_keypoints)

        if image.Laterality is None:
            print(
                f"Laterality was not set for image {image.ImageInstanceID} found laterality: {laterality}"
            )
        elif image.Laterality != laterality:
            print(f"Different laterality inferred for image {image.ImageInstanceID}")

        if image.ETDRSField is None:
            print(f"ETDRSField was not set for image {image.ImageInstanceID} found ETDRSField: {etdrs_field}")
        elif image.ETDRSField != etdrs_field:
            print(f"Different ETDRSField inferred for image {image.ImageInstanceID}")

        if update_database:
            image.Laterality = laterality
            image.ETDRSField = etdrs_field
    if update_database:
        session.commit()
    

Found 4506 color fundus images


100%|██████████| 4506/4506 [00:00<00:00, 43972.14it/s]
